Current subscription run

        ↓
deduplicate by subscription_id

        ↓
compare with trusted Silver

        ↓
INSERT / UPDATE / UNCHANGED

        ↓
subscription_cdc_work

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. Refresh and Load Bronze Subscriptions
# ============================================================

bronze_subscriptions_df = spark.table(
    "customer360_dev.bronze.subscriptions"
)


In [0]:

# ============================================================
# 2. Find Latest Incremental Run
# ============================================================

latest_run_row = (
    bronze_subscriptions_df
    .filter(
        F.col("_load_type") == "INCREMENTAL"
    )
    .filter(
        F.col("_pipeline_run_id").isNotNull()
    )
    .groupBy("_pipeline_run_id")
    .agg(
        F.max("_ingestion_timestamp").alias(
            "latest_ingestion_timestamp"
        )
    )
    .orderBy(
        F.col("latest_ingestion_timestamp").desc()
    )
    .first()
)

run_id = (
    latest_run_row["_pipeline_run_id"]
    if latest_run_row
    else None
)

print("Processing Subscription Run ID:", run_id)

Processing Subscription Run ID: 0aa31525-10e6-422c-8393-778ef3ad54c9


In [0]:
if run_id is not None:

    incremental_bronze_df = (
        bronze_subscriptions_df
        .filter(
            (F.col("_load_type") == "INCREMENTAL")
            &
            (F.col("_pipeline_run_id") == run_id)
        )
    )

else:

    incremental_bronze_df = (
        bronze_subscriptions_df.limit(0)
    )

    print(
        "No incremental subscription batch "
        "available for processing."
    )

In [0]:
incremental_count = (
    incremental_bronze_df.count()
)

print(
    f"Incremental subscription rows: "
    f"{incremental_count:,}"
)

Incremental subscription rows: 10


In [0]:
silver_subscriptions_df = spark.table(
    "customer360_dev.silver.subscriptions"
)

In [0]:
incremental_clean_df = (
    incremental_bronze_df

    .withColumn(
        "plan_code_clean",
        F.upper(F.trim("plan_code"))
    )

    .withColumn(
        "subscription_status_clean",
        F.upper(F.trim("subscription_status"))
    )

    .withColumn(
        "start_date_cast",
        F.coalesce(
            F.try_to_date("start_date", "yyyy-MM-dd"),
            F.try_to_date("start_date", "dd-MM-yyyy")
        )
    )

    .withColumn(
        "end_date_cast",
        F.coalesce(
            F.try_to_date("end_date", "yyyy-MM-dd"),
            F.try_to_date("end_date", "dd-MM-yyyy")
        )
    )

    .withColumn(
        "billing_cycle_clean",
        F.upper(F.trim("billing_cycle"))
    )

    .withColumn(
        "monthly_fee_cast",
        F.col("monthly_fee").cast("decimal(10,2)")
    )

    .withColumn(
        "auto_renew_cast",
        F.col("auto_renew").cast("boolean")
    )

    .withColumn(
        "payment_method_type_clean",
        F.upper(F.trim("payment_method_type"))
    )

    .withColumn(
        "trial_flag_cast",
        F.col("trial_flag").cast("boolean")
    )

    .withColumn(
        "cancel_reason_clean",
        F.trim("cancel_reason")
    )

    .withColumn(
        "created_at_cast",
        F.coalesce(
            F.expr("try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_timestamp(created_at, 'dd-MM-yyyy HH:mm')")
        )
    )

    .withColumn(
        "updated_at_cast",
        F.coalesce(
            F.expr("try_to_timestamp(updated_at, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_timestamp(updated_at, 'dd-MM-yyyy HH:mm')")
        )
    )
)

In [0]:
subscription_window = (
    Window
    .partitionBy("subscription_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

incremental_dedup_df = (
    incremental_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(
            subscription_window
        )
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

In [0]:
comparison_df = (
    incremental_dedup_df.alias("src")
    .join(
        silver_subscriptions_df.alias("tgt"),

        F.col("src.subscription_id")
        ==
        F.col("tgt.subscription_id"),

        how="left"
    )
)

In [0]:
change_condition = (

    ~F.col("src.customer_id").eqNullSafe(
        F.col("tgt.customer_id")
    )

    |

    ~F.col("src.plan_code_clean").eqNullSafe(
        F.col("tgt.plan_code")
    )

    |

    ~F.col("src.subscription_status_clean").eqNullSafe(
        F.col("tgt.subscription_status")
    )

    |

    ~F.col("src.start_date_cast").eqNullSafe(
        F.col("tgt.start_date")
    )

    |

    ~F.col("src.end_date_cast").eqNullSafe(
        F.col("tgt.end_date")
    )

    |

    ~F.col("src.billing_cycle_clean").eqNullSafe(
        F.col("tgt.billing_cycle")
    )

    |

    ~F.col("src.monthly_fee_cast").eqNullSafe(
        F.col("tgt.monthly_fee")
    )

    |

    ~F.col("src.auto_renew_cast").eqNullSafe(
        F.col("tgt.auto_renew")
    )

    |

    ~F.col("src.payment_method_type_clean").eqNullSafe(
        F.col("tgt.payment_method_type")
    )

    |

    ~F.col("src.trial_flag_cast").eqNullSafe(
        F.col("tgt.trial_flag")
    )

    |

    ~F.col("src.cancel_reason_clean").eqNullSafe(
        F.col("tgt.cancel_reason")
    )
)

In [0]:
classified_df = (
    comparison_df
    .withColumn(
        "cdc_action",

        F.when(
            F.col("tgt.subscription_id").isNull(),
            F.lit("INSERT")
        )

        .when(
            change_condition,
            F.lit("UPDATE")
        )

        .otherwise(
            F.lit("UNCHANGED")
        )
    )
)

In [0]:
subscription_cdc_df = (
    classified_df
    .select(
        F.col("src.subscription_id").alias(
            "subscription_id"
        ),

        F.col("src.customer_id").alias(
            "customer_id"
        ),

        F.col("src.plan_code_clean").alias(
            "plan_code"
        ),

        F.col("src.subscription_status_clean").alias(
            "subscription_status"
        ),

        F.col("src.start_date_cast").alias(
            "start_date"
        ),

        F.col("src.end_date_cast").alias(
            "end_date"
        ),

        F.col("src.billing_cycle_clean").alias(
            "billing_cycle"
        ),

        F.col("src.monthly_fee_cast").alias(
            "monthly_fee"
        ),

        F.col("src.auto_renew_cast").alias(
            "auto_renew"
        ),

        F.col("src.payment_method_type_clean").alias(
            "payment_method_type"
        ),

        F.col("src.trial_flag_cast").alias(
            "trial_flag"
        ),

        F.col("src.cancel_reason_clean").alias(
            "cancel_reason"
        ),

        F.col("src.created_at_cast").alias(
            "created_at"
        ),

        F.col("src.updated_at_cast").alias(
            "updated_at"
        ),

        F.col("src._source_file").alias(
            "_source_file"
        ),

        F.col("src._source_system").alias(
            "_source_system"
        ),

        F.col("src._load_type").alias(
            "_load_type"
        ),

        F.col("src._ingestion_timestamp").alias(
            "_ingestion_timestamp"
        ),

        F.col("src._pipeline_run_id").alias(
            "_pipeline_run_id"
        ),

        F.col("cdc_action")
    )
)

In [0]:
display(
    subscription_cdc_df.limit(50)
)

subscription_id,customer_id,plan_code,subscription_status,start_date,end_date,billing_cycle,monthly_fee,auto_renew,payment_method_type,trial_flag,cancel_reason,created_at,updated_at,_source_file,_source_system,_load_type,_ingestion_timestamp,_pipeline_run_id,cdc_action
SINC000014,CINC000014,BASIC,ACTIVE,2026-09-30,null,MONTHLY,299.00,true,CARD,false,null,2026-09-30T10:43:00.000Z,2026-09-30T10:44:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000019,CINC000019,PREMIUM,ACTIVE,2026-09-30,null,MONTHLY,799.00,true,UPI,false,null,2026-09-30T10:48:00.000Z,2026-09-30T10:49:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000017,CINC000017,BASIC,ACTIVE,2026-09-30,null,MONTHLY,299.00,true,UPI,false,null,2026-09-30T10:46:00.000Z,2026-09-30T10:47:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000020,CINC000020,BASIC,ACTIVE,2026-09-30,null,MONTHLY,299.00,true,CARD,false,null,2026-09-30T10:49:00.000Z,2026-09-30T10:50:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000018,CINC000018,STANDARD,ACTIVE,2026-09-30,null,MONTHLY,499.00,true,CARD,false,null,2026-09-30T10:47:00.000Z,2026-09-30T10:48:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000015,CINC000015,STANDARD,ACTIVE,2026-09-30,null,MONTHLY,499.00,true,UPI,false,null,2026-09-30T10:44:00.000Z,2026-09-30T10:45:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000011,CINC000011,BASIC,ACTIVE,2026-09-30,null,MONTHLY,299.00,true,UPI,false,null,2026-09-30T10:40:00.000Z,2026-09-30T10:41:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000016,CINC000016,PREMIUM,ACTIVE,2026-09-30,null,MONTHLY,799.00,true,CARD,false,null,2026-09-30T10:45:00.000Z,2026-09-30T10:46:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000012,CINC000012,STANDARD,ACTIVE,2026-09-30,null,MONTHLY,499.00,true,CARD,false,null,2026-09-30T10:41:00.000Z,2026-09-30T10:42:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT
SINC000013,CINC000013,PREMIUM,ACTIVE,2026-09-30,null,MONTHLY,799.00,true,UPI,false,null,2026-09-30T10:42:00.000Z,2026-09-30T10:43:00.000Z,dbfs:/Volumes/customer360_dev/staging/source_files/subscriptions/incremental/subscriptions_incremental_20261001.csv,SUBSCRIPTION_PLATFORM,INCREMENTAL,2026-10-01T10:29:13.882Z,0aa31525-10e6-422c-8393-778ef3ad54c9,INSERT


In [0]:
(
    subscription_cdc_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.control.subscription_cdc_work"
    )
)